In [101]:
import torch
import torchcodec
from datasets import load_dataset

ds = load_dataset("rodriler/isolated-guitar-chords")

## Functions for feature extraction

(Add to their own pre-processing package later)

In [131]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

def extract_chroma_cqt(sample):
    """Extract chroma CQT from a HuggingFace dataset sample.
    
    Args:
        sample: HuggingFace dataset row with sample["audio"] 
                as a torchcodec AudioDecoder object
    
    Returns:
        chroma: np.ndarray of shape (12, T)
    """
    audio = sample["audio"].get_all_samples()
    waveform = audio.data.cpu().numpy()
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=0)   # convert stereo/multi-channel to mono
    else:
        waveform = waveform.squeeze()
    sr = audio.sample_rate

    # Resample to our standard rate if needed
    if sr != 22050:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=22050)
        sr = 22050

    # Harmonic separation — reduce pick/percussion noise
    y_harmonic = librosa.effects.harmonic(waveform, margin=1.0)

    # Chroma CQT
    chroma = librosa.feature.chroma_cqt(
        y=y_harmonic,
        sr=sr,
        hop_length=512,
        fmin=32.7,
        n_chroma=12,
        bins_per_octave=12
    )

    return chroma

def slice_into_windows(chroma, context_frames=15):
    if chroma.ndim != 2:
        raise ValueError(f"Expected chroma shape (12, T), got {chroma.shape}")

    n_frames = chroma.shape[1]

    if n_frames < context_frames:
        necessary_padding = context_frames - n_frames
        padded = np.pad(chroma, ((0, 0), (0, necessary_padding)))
        return padded[np.newaxis, :]

    windows = []
    for i in range(n_frames - context_frames + 1):
        window = chroma[:, i:i + context_frames]
        windows.append(window)

    return np.array(windows)

## Prepare Data

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import numpy as np

class ChordDataset(Dataset):
    def __init__(self, samples):
        """
        Args:
            samples: HuggingFace dataset split (e.g., ds["train"])
        """
        all_windows = []
        all_labels = []

        for sample in samples:
            chroma = extract_chroma_cqt(sample)
            windows = slice_into_windows(chroma)
            label = sample["label"]
            all_windows.append(windows)
            all_labels.extend([label] * len(windows))

        all_windows = np.concatenate(all_windows, axis=0)

        self.features = torch.FloatTensor(all_windows).unsqueeze(1)  # (N, 1, 12, 15)
        self.labels = torch.LongTensor(all_labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


train_dataset = ChordDataset(ds["train"])
test_dataset = ChordDataset(ds["test"])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



## Define Starter Model

In [ ]:
class ChordCNN(nn.Model):
    def __init__(self, num_classes=8):
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1
        )
        self.bn1 = nn.BatchNorm2d(32)
        self.drop1 = nn.Dropout2d(p=0.15)
        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )
 
        self.bn2 = nn.BatchNorm2d(64)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
 
        self.drop2 = nn.Dropout2d(p=0.15)

        self.gap = nn.AdaptiveAvgPool2d(output_size=(1, 1))

        self.fc1 = nn.Linear(64, 32)
        self.drop_fc = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(32, num_classes)



